# Topic 12 — Practice: Capstone Investigation

**The notebook is the lesson.** Work top to bottom. Cells with `assert` grade themselves — a silent run = pass, an `AssertionError` = fix your code. Warm-Up, Interview Lens and Reflection have **no answer key** — answer in writing.

_Spaced repetition: ~60% of this notebook is the current topic, ~40% revisits earlier topics._

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
products['list_price'] = np.where(products['list_price']<0, np.nan, products['list_price'])
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
orders['order_date'] = pd.to_datetime(orders['order_date'], format='mixed', dayfirst=True, errors='coerce')
orders['month'] = orders['order_date'].dt.to_period('M').astype(str)
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
master = (items.merge(products[['product_id','category','unit_cost']], on='product_id', how='left')
              .merge(orders[['order_id','channel','status','month','order_date']], on='order_id', how='left'))
master['line_cogs'] = master['quantity']*master['unit_cost']
master['line_profit'] = master['line_revenue'] - master['line_cogs']
returns = pd.read_csv(RAW+'returns.csv', dtype={'order_id':str})


## 🔁 Warm-Up Recall (earlier topics — no answers given)

Final recall across the whole course:

1. (T04) Name three cleaning fixes Aurora's data needed.
2. (T06) What invariant proves a join didn't fan out?
3. NumPy: how do you compute a weighted ratio ignoring zero-denominator rows?

In [ ]:
# Final NumPy recall: overall margin ignoring zero-revenue rows
rev = np.array([100., 200., 0., 50.]); cogs = np.array([60., 120., 0., 25.])
mask = rev > 0
margin = ...   # TODO (rev-cogs).sum()/rev.sum() over mask
assert round(float(margin),4) == round(float((rev[mask]-cogs[mask]).sum()/rev[mask].sum()),4)
print('ok')

## 🎯 Core Lesson Tasks (current topic)

This is the job: assemble everything into a defensible report. Objective KPIs self-check; the narrative has no key.

**Core 1 — grain-safe master.** Confirm the master table preserved the line grain.

In [ ]:
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
assert len(master) == len(items), 'a join changed the grain!'
print('master rows:', len(master))

**Core 2 — the KPIs.** Compute gross & net revenue, AOV, returns rate, gross margin. All self-checked.

In [ ]:
gross = master['line_revenue'].sum()
total_refunds = returns['refund_amount'].sum()
net = ...
n_orders = orders['order_id'].nunique()
aov = ...
returns_rate = (orders['status']=='returned').mean()
gross_margin = ...
assert net == gross - total_refunds
assert 0 <= returns_rate <= 1 and 0 <= gross_margin <= 1
print(f'gross £{gross:,.0f} | net £{net:,.0f} | AOV £{aov:,.2f} | returns {returns_rate:.1%} | margin {gross_margin:.1%}')

**Core 3 — resolve the dip.** Plot monthly revenue + 3mo MA and state your verdict (real vs artifact).

In [ ]:
monthly = master.dropna(subset=['order_date']).set_index('order_date').sort_index()['line_revenue'].resample('ME').sum()
ax = monthly.plot(figsize=(11,4)); monthly.rolling(3).mean().plot(ax=ax)
ax.set_title('Verdict: real dip or seasonal artifact? (write it)'); ax.set_ylabel('Revenue £'); plt.show()
assert len(monthly) >= 12
print('done')

## 🔀 Mixed Review Tasks (earlier topics — spaced repetition)

**Mixed review (Topics 08/09) — corroborate with text + features.** Count late & damaged support mentions.

In [ ]:
tickets = pd.read_csv(RAW+'support_tickets.csv', dtype={'ticket_id':str})
m = tickets['message'].str.lower()
late = ...
damaged = ...
assert late > 0 and damaged > 0
print('late', late, 'damaged', damaged)

**Mixed review (Topic 10) — management cross-tab.** Revenue by channel × month with margins.

In [ ]:
pivot = ...
assert 'All' in pivot.index
pivot.iloc[:, :3]

## 🕵️ Data Detective Investigation

**The case closes — final report.** Answer the brief with evidence: (1) is revenue actually falling? (2) where is margin leaking? (3) top 3 actions, each tied to a number. Build the one-page dashboard and write the narrative below (no key — you defend it like a real analyst).

In [ ]:
channel_rev = master.groupby('channel')['line_revenue'].sum().sort_values(ascending=False)
monthly = master.dropna(subset=['order_date']).set_index('order_date').sort_index()['line_revenue'].resample('ME').sum()
fig, axes = plt.subplots(2,2, figsize=(13,8))
monthly.plot(ax=axes[0,0], title='Monthly revenue')
channel_rev.plot(kind='bar', ax=axes[0,1], title='Revenue by channel')
master.groupby('category')['line_profit'].sum().sort_values().plot(kind='barh', ax=axes[1,0], title='Profit by category')
master.groupby('channel')['line_profit'].sum().plot(kind='bar', ax=axes[1,1], title='Profit by channel')
fig.tight_layout()
assert len(fig.axes) >= 4
plt.show()
print('Now write the executive summary + 3 recommendations.')

## 🔎 Interview Lens (write answers — no model answers)

- **Q30:** Defend your headline revenue number to the Head of Finance — trace it to raw rows.
- **Q32:** What does *data quality* mean, concretely, on a dataset like Aurora's?

## ✍️ Reflection (write short explanations)

1. Re-answer Q30 and Q32 as a presentation.
2. Write your ≤150-word executive summary.
3. List your top-3 recommendations, each tied to a specific number.